In [1]:
import pandas as pd
import pickle
import math
import re

In [2]:
with open("../Lab4/unigram_model.pkl", "rb") as f:
    unigram_counts = pickle.load(f)

with open("../Lab4/bigram_model.pkl", "rb") as f:
    bigram_counts = pickle.load(f)

print("Models loaded (unigram size:", len(unigram_counts), 
      ", bigram size:", len(bigram_counts), ")")

Models loaded (unigram size: 558957 , bigram size: 8242038 )


In [18]:
def get_count(model, key):
    # Unigram: wrap in tuple if string
    if isinstance(key, str):
        key = (key,)
    value = model.get(key, 0)

    if isinstance(value, int):
        return value
    if isinstance(value, dict):
        return value.get("count", 0)
    if isinstance(value, tuple):
        return value[0]
    return 0

In [19]:
N = sum(get_count(unigram_counts, k) for k in unigram_counts.keys())

In [20]:
def extract_tokens(df):
    tokens = []
    for row in df["sentences"]:
        if isinstance(row, (list, tuple)) or hasattr(row, "__iter__"):
            for item in row:
                if isinstance(item, dict) and "tokens" in item:
                    tokens.extend([t for t in item["tokens"] if isinstance(t, str)])
    return tokens


In [21]:
val_df = pd.read_parquet("../Lab5/validation.parquet")
test_df = pd.read_parquet("../Lab5/test.parquet")

In [31]:
val_tokens = [t for sent in val_df["sentences"] for t in str(sent).split()]
test_tokens = [t for sent in test_df["sentences"] for t in str(sent).split()]

print("Validation tokens:", len(val_tokens))
print("Test tokens:", len(test_tokens))

Validation tokens: 126116
Test tokens: 125633


In [32]:
def pmi(w1, w2):
    # Wrap unigram keys in tuple
    c_bigram = get_count(bigram_counts, (w1, w2))
    if c_bigram == 0:
        return float("-inf")
    c_w1 = get_count(unigram_counts, (w1,))   # <-- wrap as tuple
    c_w2 = get_count(unigram_counts, (w2,))   # <-- wrap as tuple
    return math.log((c_bigram * N) / (c_w1 * c_w2 + 1e-10))


In [33]:
def compute_pmi_table(tokens, split_name="Validation"):
    bigrams = list(zip(tokens[:-1], tokens[1:]))
    records = []
    for w1, w2 in bigrams:
        records.append({
            "Bigram": (w1, w2),
            "Count_bigram": get_count(bigram_counts, (w1, w2)),
            "Count_w1": get_count(unigram_counts, w1),
            "Count_w2": get_count(unigram_counts, w2),
            "PMI": pmi(w1, w2)
        })

    df = pd.DataFrame(records).drop_duplicates(subset=["Bigram"])
    print(f"\nTop 10 PMI scores for {split_name} set:")
    display(df.sort_values("PMI", ascending=False).head(10))
    return df

In [34]:
val_pmi  = compute_pmi_table(val_tokens, "Validation")
test_pmi = compute_pmi_table(test_tokens, "Test")


Top 10 PMI scores for Validation set:


,Bigram,Count_bigram,Count_w1,Count_w2,PMI
126109,"('पर',, 'ला',)",0,0,0,-inf
0,"([{'text':, 'होम)",0,0,0,-inf
1,"('होम, स्क्रीन)",0,0,0,-inf
126081,"(क्रोएशिया, को)",0,0,0,-inf
126080,"(कर, क्रोएशिया)",0,0,0,-inf
126079,"(गोल, कर)",0,0,0,-inf
126078,"(में, गोल)",0,0,0,-inf
126076,"(28वें, मिनट)",0,0,0,-inf
126075,"(ने, 28वें)",0,0,0,-inf
126074,"(पेरीसिच, ने)",0,0,0,-inf



Top 10 PMI scores for Test set:


,Bigram,Count_bigram,Count_w1,Count_w2,PMI
125630,"('क्रिया'],, dtype=object)})",0,0,0,-inf
0,"([{'text':, 'दोनों)",0,0,0,-inf
1,"('दोनों, बल्लेबाजों)",0,0,0,-inf
125608,"('आकस्मिक',, 'जिम',)",0,0,0,-inf
125607,"('स्वेटर',, 'आकस्मिक',)",0,0,0,-inf
125606,"('की',, 'स्वेटर',)",0,0,0,-inf
125605,"('पुरुषों',, 'की',)",0,0,0,-inf
125604,"('कस्टम',, 'पुरुषों',)",0,0,0,-inf
125603,"('सादे',, 'कस्टम',)",0,0,0,-inf
125602,"('अद्वितीय',, 'सादे',)",0,0,0,-inf


In [ ]:
val_pmi.to_parquet("validation_pmi.parquet", index=False)
test_pmi.to_parquet("test_pmi.parquet", index=False)

print("\nPMI tables saved")

In [26]:
print("Example val token:", val_tokens[0])
print("As tuple:", (val_tokens[0],))
print("Unigram exists in model:", (val_tokens[0],) in unigram_counts)
print("Raw string exists:", val_tokens[0] in unigram_counts)


Example val token: होम
As tuple: ('होम',)
Unigram exists in model: True
Raw string exists: False


In [27]:
print("Example val bigram:", (val_tokens[0], val_tokens[1]))
print("Exists in bigram model:", (val_tokens[0], val_tokens[1]) in bigram_counts)


Example val bigram: ('होम', 'स्क्रीन')
Exists in bigram model: True
